In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from collections import Counter

from torch.utils.data import TensorDataset, DataLoader

In [8]:
train = pd.read_csv("/content/train.csv")

## Create MCQ Pairs

In [9]:
def create_mcq_pairs(df):
    rows = []
    option_columns = ["A", "B", "C", "D", "E"]
    for _, row in df.iterrows():
        for option in option_columns:
            rows.append({
                "id": row["id"],
                "prompt": row["prompt"],
                "option": row[option],
                "option_id": option,
                "label": int(option == row["answer"])
            })
    return pd.DataFrame(rows)

## Prepare Data

In [10]:
bilstm_train_q, bilstm_valid_q = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["answer"]
)

bilstm_train = create_mcq_pairs(bilstm_train_q)
bilstm_valid = create_mcq_pairs(bilstm_valid_q)

def tokenize_text(text):
    return str(text).lower().split()

words = Counter()

for _, row in bilstm_train.iterrows():
    words.update(tokenize_text(row["prompt"] + " " + row["option"]))

vocab = {"<PAD>": 0, "<UNK>": 1}

for word in words:
    vocab[word] = len(vocab)

MAX_LEN = 128

def encode(text):
    ids = [vocab.get(w, 1) for w in tokenize_text(text)]
    ids = ids[:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X_train = torch.tensor(
    [encode(r["prompt"] + " " + r["option"])
     for _, r in bilstm_train.iterrows()]
)

X_valid = torch.tensor(
    [encode(r["prompt"] + " " + r["option"])
     for _, r in bilstm_valid.iterrows()]
)

y_train = torch.tensor(bilstm_train["label"].values)
y_valid = torch.tensor(bilstm_valid["label"].values)

print("Vocabulary size:", len(vocab))
print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

Vocabulary size: 3815
Train: torch.Size([8000, 128])
Validation: torch.Size([2000, 128])


In [11]:
train_loader = DataLoader(TensorDataset(X_train, y_train),batch_size=32,shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid),batch_size=32,shuffle=False)

## Model

In [12]:
class BiLSTM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)

        self.lstm = nn.LSTM(64,64,batch_first=True,bidirectional=True)

        self.fc = nn.Linear(128, 2)

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat((hidden[-2], hidden[-1]),dim=1)

        return self.fc(hidden)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bilstm_model = BiLSTM(len(vocab)).to(device)

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 4.0]).to(device))

optimizer = torch.optim.Adam(bilstm_model.parameters(),lr=0.001)

print(bilstm_model)

BiLSTM(
  (embedding): Embedding(3815, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


## Training

In [13]:
for epoch in range(8):

    bilstm_model.train()
    train_loss = 0

    for X, y in train_loader:

        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        loss = criterion(bilstm_model(X), y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    bilstm_model.eval()

    val_loss = 0
    preds, labels = [], []

    with torch.no_grad():

        for X, y in valid_loader:

            X, y = X.to(device), y.to(device)

            output = bilstm_model(X)

            val_loss += criterion(output, y).item()

            preds.extend(
                output.argmax(1).cpu().numpy()
            )

            labels.extend(
                y.cpu().numpy()
            )

    train_loss /= len(train_loader)
    val_loss /= len(valid_loader)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {acc:.4f} | "
        f"F1: {f1:.4f}"
    )

Epoch 1 | Train Loss: 0.6947 | Val Loss: 0.6920 | Accuracy: 0.8015 | F1: 0.0149
Epoch 2 | Train Loss: 0.6930 | Val Loss: 0.6912 | Accuracy: 0.8015 | F1: 0.0149
Epoch 3 | Train Loss: 0.6914 | Val Loss: 0.6894 | Accuracy: 0.7025 | F1: 0.1883
Epoch 4 | Train Loss: 0.6913 | Val Loss: 0.6894 | Accuracy: 0.8015 | F1: 0.0149
Epoch 5 | Train Loss: 0.6908 | Val Loss: 0.6886 | Accuracy: 0.2045 | F1: 0.3346
Epoch 6 | Train Loss: 0.6959 | Val Loss: 0.6904 | Accuracy: 0.8015 | F1: 0.0149
Epoch 7 | Train Loss: 0.6923 | Val Loss: 0.6898 | Accuracy: 0.5670 | F1: 0.2661
Epoch 8 | Train Loss: 0.6912 | Val Loss: 0.6896 | Accuracy: 0.8015 | F1: 0.0149


## Validation Using MAP@3

In [14]:
bilstm_model.eval()

scores = []

with torch.no_grad():

    for X, _ in valid_loader:

        output = bilstm_model(X.to(device))

        scores.extend(torch.softmax(output, dim=1)[:, 1].cpu().numpy())

bilstm_results = bilstm_valid.copy()
bilstm_results["score"] = scores

bilstm_top3 = (
    bilstm_results
    .sort_values(["id", "score"], ascending=[True, False])
    .groupby("id")
    .head(3)
)

bilstm_predictions = (
    bilstm_top3
    .groupby("id")["option_id"]
    .apply(list)
    .to_dict()
)

bilstm_truth = (
    bilstm_valid[bilstm_valid["label"] == 1]
    .set_index("id")["option_id"]
    .to_dict()
)

bilstm_map3 = np.mean([
    (
        1 / (bilstm_predictions[q].index(bilstm_truth[q]) + 1)
        if bilstm_truth[q] in bilstm_predictions[q][:3]
        else 0
    )
    for q in bilstm_truth
])

print("BiLSTM MAP@3 =", round(bilstm_map3, 4))

BiLSTM MAP@3 = 0.3854
